In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from mlflow.models.signature import infer_signature

In [0]:
ml_features_df = spark.read.table("finops_catalog_2.finops.ml_features")


### Convert to Pandas

In [0]:
pdf = ml_features_df.toPandas()

X = pdf[
    [   
        "net_cost",
        "usage_quantity",
        "anomaly_score",
        "budget_amount",
        "budget_utilization_pct",
        "cost_variance_7d_pct",
        "cost_variance_30d_pct",
        "cost_budget_pct",
        "cost_per_unit",
        "total_savings",
        "high_budget_flag",
        "high_variance_flag"
    ]
]

X = X.fillna(0)

In [0]:
mlflow.set_experiment("/Users/omkarmhetre800@gmail.com/FinOps_CloudBudget_AnomalyDetection")

In [0]:
with mlflow.start_run(run_name="isolation_forest_budget_anomaly") as run:

    model = IsolationForest(
        n_estimators=200,
        contamination=0.01,
        random_state=42
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipeline.fit(X)

    preds = pipeline.predict(X)
    anomaly_flags = np.where(preds == -1, 1, 0)

    pdf["pred_anomaly"] = anomaly_flags

    anomaly_rate = anomaly_flags.mean()

    mlflow.log_metric("anomaly_rate", float(anomaly_rate))

    X_scaled = pipeline.named_steps["scaler"].transform(X)
    score = silhouette_score(X_scaled, anomaly_flags)
    mlflow.log_metric("silhouette_score", float(score))


    mlflow.log_param("model_type", "IsolationForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("contamination", 0.01)

    input_example = X[:5]
    signature = infer_signature(X, anomaly_flags)

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="model",
        input_example=input_example,
        signature=signature
    )

    model_uri = f"runs:/{run.info.run_id}/model"

    mlflow.register_model(
        model_uri,
        "finops_cloud_budget_anomaly_model"
    )

In [0]:
spark_df = spark.createDataFrame(pdf)

spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "finops_catalog_2.finops.ml_predictions"
    )


In [0]:
from pyspark.sql.functions import col
spark_df.filter(col("pred_anomaly") == 1).count() # 2741

spark_df.groupBy("pred_anomaly") .count().show()


### Load & Validate Model

In [0]:
loaded_model = mlflow.sklearn.load_model(model_uri)

test_preds = loaded_model.predict(X[:10])
test_preds  # array([-1,  1,  1,  1,  1,  1,  1,  1,  1,  1])

In [0]:
display(X.iloc[0])